# Restroom 모범답안 (2단계 CLIP 매칭) — 실제 캐글 대회 제출 (Colab)

IOAI 2025 Individual **Restroom**(화장실 아이콘 이성 매칭) *모범답안*의 코드를 **그대로** 옮겨, 사설 캐글 대회
[**restroom-ioai-2025**](https://www.kaggle.com/competitions/restroom-ioai-2025) 에서 다운로드·학습·제출한다.

**기법** (모범답안 2단계 CLIP 미세조정):
1. **crop-matcher** (`crop_model`): 잘린(crop) 아이콘 ↔ 같은 성별 원본(orig) 을 InfoNCE 로 매칭(잘린 query 의 원본 찾기).
2. **gender-matcher** (`gender_model`): 같은 화장실의 **남↔여 원본**을 InfoNCE 로 매칭(이성 짝 찾기).
추론: query 잘린아이콘 → crop_model 로 같은성별 원본 찾고 → gender_model 로 **반대 성별 같은 화장실** 원본 선택.

**구성** (모범답안 그대로): `CLIPReID`(CLIP ViT-B/32 마지막 블록 미세조정) · `CropMatchDataset`/`train_crop_matcher` ·
`GenderMatchDataset`/`train_gender_matcher`(InfoNCE).

> **제출 형식**: 대회 baseline 과 동일하게 `submission.csv`(`id,array`) — `array` 는 query(파일번호 정렬) 별 매칭된
> gallery 의 **1-기반 위치**(파일번호 정렬) 리스트의 문자열. (대회 test 는 파일명이 섞여 있어 ID 가 아닌 *정렬 위치* 로 채점.)
> ⚙️ GPU 필요(CLIP 미세조정 ×2). T4 수 분. ⚠️ API 토큰 평문 — 외부 공유 금지.

## 0. 설치 · CLIP · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "pillow", "ftfy", "regex", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 restroom-ioai-2025 다운로드
403 이면 [대회 페이지](https://www.kaggle.com/competitions/restroom-ioai-2025/rules)에서 'I Understand and Accept' 1회 후 재실행.

In [ ]:
import glob, zipfile
from pathlib import Path
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("restroom-ioai-2025", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료")
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:200])
    print("→ 403 이면 https://www.kaggle.com/competitions/restroom-ioai-2025/rules 에서 수락 후 재실행")
def _find(*cs):
    for c in cs:
        if os.path.isdir(c): return c
    return cs[0]
train_dir = Path(_find(os.path.join(WORK_DIR, "training_set/training_set"), os.path.join(WORK_DIR, "training_set")))
TEST = Path(_find(os.path.join(WORK_DIR, "test_set/test_set"), os.path.join(WORK_DIR, "test_set")))
print("train:", train_dir, "| test:", TEST, "| query:", len(glob.glob(str(TEST/'query/*.png'))), "gallery:", len(glob.glob(str(TEST/'gallery/*.png'))))

## 2. CLIPReID — CLIP ViT-B/32 마지막 블록 미세조정 (모범답안 그대로)

In [ ]:
import clip
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from tqdm import tqdm

class CLIPReID(nn.Module):
    def __init__(self, unfreeze_last_n_layers=1):
        super().__init__()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model, self.preprocess = clip.load("ViT-B/32", device=self.device)
        for p in self.clip_model.parameters(): p.requires_grad = False
        total = len(self.clip_model.visual.transformer.resblocks)
        for i in range(total - unfreeze_last_n_layers, total):
            for p in self.clip_model.visual.transformer.resblocks[i].parameters(): p.requires_grad = True
        for p in self.clip_model.visual.ln_post.parameters(): p.requires_grad = True
    def forward_one(self, x):
        x = x.to(dtype=torch.float32, device=self.device)
        if self.training: feats = self.clip_model.encode_image(x)
        else:
            with torch.no_grad(): feats = self.clip_model.encode_image(x)
        return F.normalize(feats.to(dtype=torch.float32), p=2, dim=1)

## 3. 1단계 crop-matcher (잘린↔같은성별 원본, InfoNCE) (모범답안 그대로)

In [ ]:
class CropMatchDataset(Dataset):
    def __init__(self, data_dir):
        _, self.preprocess = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")
        n = len(os.listdir(os.path.join(data_dir, 'crop', 'male')))
        self.pairs = [(os.path.join(data_dir,'orig','male',f'{i}.png'), os.path.join(data_dir,'crop','male',f'{i}.png')) for i in range(1,n+1)] + \
                     [(os.path.join(data_dir,'orig','female',f'{i}.png'), os.path.join(data_dir,'crop','female',f'{i}.png')) for i in range(1,n+1)]
        print(f"Found {len(self.pairs)} matching pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        op, cp = self.pairs[idx]
        return self.preprocess(Image.open(op).convert('RGB')), self.preprocess(Image.open(cp).convert('RGB'))

def train_crop_matcher(train_dir, epochs=50, batch_size=32, unfreeze_last_n_layers=6):
    loader = DataLoader(CropMatchDataset(str(train_dir)), batch_size=batch_size, shuffle=True, num_workers=0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPReID(unfreeze_last_n_layers).to(dev).float()
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5, betas=(0.5,0.999))
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5); temp = 0.07; best = float('inf')
    for ep in range(epochs):
        model.train(); tot = 0
        for orig, crop in loader:
            opt.zero_grad()
            of = F.normalize(model.forward_one(orig.float().to(dev)), dim=1)
            cf = F.normalize(model.forward_one(crop.float().to(dev)), dim=1)
            logits = (of @ cf.t()) / temp; lab = torch.arange(of.size(0), device=dev)
            loss = 0.5*(F.cross_entropy(logits, lab) + F.cross_entropy(logits.t(), lab))
            loss.backward(); opt.step(); tot += loss.item()
        sched.step(); avg = tot/len(loader)
        if avg < best: torch.save(model.state_dict(), 'crop_model.pth'); best = avg
    print(f"crop-matcher done, best loss {best:.4f}"); return model

crop_model = train_crop_matcher(train_dir)

## 4. 2단계 gender-matcher (같은 화장실 남↔여 원본, InfoNCE) (모범답안 그대로)

In [ ]:
class GenderMatchDataset(Dataset):
    def __init__(self, data_dir):
        _, self.preprocess = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")
        n = len(os.listdir(os.path.join(data_dir, 'crop', 'male')))
        self.pairs = [(os.path.join(data_dir,'orig','male',f'{i}.png'), os.path.join(data_dir,'orig','female',f'{i}.png')) for i in range(1,n+1)]
        print(f"Found {len(self.pairs)} gender pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        mp, fp = self.pairs[idx]
        return self.preprocess(Image.open(mp).convert('RGB')), self.preprocess(Image.open(fp).convert('RGB'))

def train_gender_matcher(train_dir, epochs=50, batch_size=32, unfreeze_last_n_layers=6):
    loader = DataLoader(GenderMatchDataset(str(train_dir)), batch_size=batch_size, shuffle=True, num_workers=0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPReID(unfreeze_last_n_layers).to(dev).float()
    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5, betas=(0.5,0.999))
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6); temp = 0.07; best = float('inf')
    for ep in range(epochs):
        model.train(); tot = 0
        for male, female in loader:
            opt.zero_grad()
            mf = F.normalize(model.forward_one(male.float().to(dev)), dim=1)
            ff = F.normalize(model.forward_one(female.float().to(dev)), dim=1)
            logits = (mf @ ff.t()) / temp; lab = torch.arange(mf.size(0), device=dev)
            loss = 0.5*(F.cross_entropy(logits, lab) + F.cross_entropy(logits.t(), lab))
            loss.backward(); opt.step(); tot += loss.item()
        sched.step(); avg = tot/len(loader)
        if avg < best: torch.save(model.state_dict(), 'gender_model.pth'); best = avg
    print(f"gender-matcher done, best loss {best:.4f}"); return model

gender_model = train_gender_matcher(train_dir)

## 5. 추론 & 제출 — 2단계 매칭 → submission.csv (대회 형식 `id,array`)
query·gallery 를 **파일번호로 정렬**하고, 모범답안의 2단계 매칭으로 각 query 의 짝을 찾아 **gallery 의 1-기반 정렬 위치**
리스트를 `array` 에 저장(대회 baseline 형식).

In [ ]:
import pandas as pd
filenum = lambda p: int(os.path.splitext(os.path.basename(p))[0])

def embed(model, paths):
    model.eval(); out = []
    for p in paths:
        x = model.preprocess(Image.open(p).convert('RGB')).unsqueeze(0).float().to(model.device)
        out.append(model.forward_one(x))
    return torch.cat(out)

q_paths = sorted(glob.glob(str(TEST/'query/*.png')), key=filenum)
g_paths = sorted(glob.glob(str(TEST/'gallery/*.png')), key=filenum)
qc = embed(crop_model, q_paths); gc = embed(crop_model, g_paths)     # 1단계: crop 관계 특징
gg = embed(gender_model, g_paths)                                    # 2단계: gender 특징(gallery)

results = []; matched = set()
for i in range(len(q_paths)):
    crop_idx = int((qc[i:i+1] @ gc.t()).argmax())                    # 같은 성별 원본 찾기
    sims = (gg[crop_idx:crop_idx+1] @ gg.t())[0]                     # 그 원본의 이성 짝 찾기
    best, bs = -1, -1e9
    for gi in range(len(g_paths)):
        if gi == crop_idx or gi in matched: continue
        if float(sims[gi]) > bs: bs = float(sims[gi]); best = gi
    matched.add(best); results.append(best + 1)                      # 1-기반 gallery 위치

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": [0], "array": [str(results)]}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| query:", len(results), "| 예시:", results[:8])

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [**restroom-ioai-2025 제출 페이지**](https://www.kaggle.com/competitions/restroom-ioai-2025/submit) 에 업로드.

Restroom **모범답안**(2단계 CLIP 미세조정: crop-matcher + gender-matcher)을 그대로 옮긴 버전. 추론은 *query→같은성별 원본
→이성 짝* 의 2단계 매칭. 더: unfreeze 층/에폭↑, 증강, 앙상블. (대회: https://www.kaggle.com/competitions/restroom-ioai-2025)